# Data Mining - Homework 1


In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").getOrCreate()
sc = spark.sparkContext


## Q1
2-head word의 빈도를 구한 뒤, 빈도가 32 미만인 항목들의 빈도 합을 출력한다.


In [8]:
import urllib.request

# URL에서 데이터를 직접 읽어와 RDD로 병렬화합니다.
url = "https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/quiz/sherlock.txt"
with urllib.request.urlopen(url) as response:
    html = response.read().decode('utf-8')

# 줄 단위로 분리하여 RDD 생성
q1 = sc.parallelize(html.splitlines())

answer1 = q1.flatMap(lambda line: line.split(" ")) \
    .filter(lambda word: len(word) >= 2) \
    .map(lambda word: (word[:2].lower(), 1)) \
    .reduceByKey(lambda a, b: a + b) \
    .filter(lambda x: x[1] < 32) \
    .map(lambda x: x[1]) \
    .reduce(lambda a, b: a + b)

print(answer1)

830


## Q2
각 정수 쌍의 첫 번째 원소를 이용해 첫 값, 최댓값, 최솟값, 1의 개수, 2의 개수를 구하고 모두 더한다.


In [6]:
import urllib.request

# URL에서 데이터를 직접 읽어와 RDD로 병렬화합니다.
url_q2 = "https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/quiz/int-pair-input.txt"
with urllib.request.urlopen(url_q2) as response:
    data_q2 = response.read().decode('utf-8')

q2 = sc.parallelize(data_q2.splitlines())

first_values = q2.map(lambda line: int(line[1:-1].split(", ")[0]))

first_value = first_values.first()
max_value = first_values.reduce(lambda a, b: a if a > b else b)
min_value = first_values.reduce(lambda a, b: a if a < b else b)
count_1 = first_values.filter(lambda x: x == 1).count()
count_2 = first_values.filter(lambda x: x == 2).count()

answer2 = first_value + max_value + min_value + count_1 + count_2

print(answer2)

93


## Q3
2015~2019년 행복지수가 매년 이전 연도보다 크거나 같은 국가를 출력한다.


In [7]:
import urllib.request
import csv
import io

base = "https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/data/World%20Hapiness%20Report/"

def load_rdd_from_url(url):
    with urllib.request.urlopen(url) as response:
        content = response.read().decode('utf-8')
    # CSV 파싱을 정확하게 처리하기 위해 csv.reader 사용
    lines = content.splitlines()
    # 헤더 제외
    reader = csv.reader(lines)
    header = next(reader)
    data = list(reader)
    return sc.parallelize(data)

y2015 = load_rdd_from_url(base + "2015.csv") \
    .map(lambda x: (x[0], float(x[2])))

y2016 = load_rdd_from_url(base + "2016.csv") \
    .map(lambda x: (x[0], float(x[2])))

y2017 = load_rdd_from_url(base + "2017.csv") \
    .map(lambda x: (x[0], float(x[2])))

y2018 = load_rdd_from_url(base + "2018.csv") \
    .map(lambda x: (x[0], float(x[2])))

y2019 = load_rdd_from_url(base + "2019.csv") \
    .map(lambda x: (x[0], float(x[2])))

scores = y2015.join(y2016) \
    .mapValues(lambda x: (x[0], x[1])) \
    .join(y2017) \
    .mapValues(lambda x: (x[0][0], x[0][1], x[1])) \
    .join(y2018) \
    .mapValues(lambda x: (x[0][0], x[0][1], x[0][2], x[1])) \
    .join(y2019) \
    .mapValues(lambda x: (x[0][0], x[0][1], x[0][2], x[0][3], x[1]))

answer3 = scores.filter(
    lambda x: x[1][0] <= x[1][1]
    and x[1][1] <= x[1][2]
    and x[1][2] <= x[1][3]
    and x[1][3] <= x[1][4]
).map(lambda x: x[0]).collect()

print(answer3)

['Gabon', 'Malta', 'Senegal', 'Chad', 'Czech Republic', 'Cambodia', 'Tajikistan', 'Congo (Brazzaville)', 'Latvia', 'Finland', 'Dominican Republic', 'Honduras', 'Syria', 'Estonia', 'Mongolia', 'Poland', 'Togo', 'Slovakia', 'Hungary', 'Cameroon', 'Romania', 'Benin', 'Burkina Faso', 'Serbia', 'Ivory Coast', 'Philippines', 'Portugal', 'Niger']
